In [35]:
import os, rasterio, sys, rioxarray, pyflwdir, shutil, shapely
sys.path.append('backend/app/')
from rasterio.io import MemoryFile
from netCDF4 import Dataset
import geopandas as gpd, pandas as pd
import numpy as np, xarray as xr
from backend.app.services import flow_functions
from shapely.geometry import Polygon, MultiPolygon
from shapely import force_2d
from pyflwdir import dem
from scipy.ndimage import sobel
from hydromt_wflow import WflowSbmModel
from tqdm import tqdm
from owslib.wcs import WebCoverageService
from shapely.geometry import shape
from shapely.ops import unary_union
from rasterio.features import shapes
np.random.seed(42)

## Child Functions

In [ ]:
def create_forcing(time, ny, nx, values, single_value=True):
    if single_value:
        data = np.empty((len(time), ny, nx), dtype=np.float32)
        data[:] = values[:, None, None]
    

    return data

In [61]:
sample_folder, test_folder = 'inputs', 'test'
catchment_path = os.path.join(sample_folder, 'catchment.geojson')
terrain_path = os.path.join(sample_folder, 'dtm10.tif')
soil_path = os.path.join(sample_folder, 'soil.geojson')
land_path = os.path.join(sample_folder, 'land.geojson')
river_path = os.path.join(sample_folder, 'river.geojson')
terrain = rioxarray.open_rasterio(terrain_path).squeeze()
catchment = gpd.read_file(catchment_path)
initial_param = [0.25,0.3,50,0,0,500,0,0,100]
land = gpd.read_file(land_path)
river = gpd.read_file(river_path)

In [65]:
from whitebox.whitebox_tools import WhiteboxTools
wtb = WhiteboxTools()
wtb.set_verbose_mode(True)

with rasterio.open(terrain_path) as src:
    dem_array = src.read(1).astype("float32")
    transform, crs = src.transform, src.crs
    profile = src.profile

wtb.fill_depressions(dem=r'C:\Users\vanln\Downloads\Hydro-AI-Platform\inputs\dtm10.tif',
                      output=r'C:\Users\vanln\Downloads\Hydro-AI-Platform\filled.tif')






# lat, lon = 62.458711715169, 6.355332891871
# gdf = gpd.GeoDataFrame(geometry=[shapely.geometry.Point(lon, lat)], crs="EPSG:4326")
# x, y = gdf.to_crs(crs).geometry.iloc[0].coords[0]
# flw = pyflwdir.from_dem(dem_array, transform=transform)
# flow_acc = flw.accuflux(np.ones_like(dem_array))
# mask = flow_acc > 50
# row, col = flw.snap(xy=(x, y), mask=mask, max_length=10)
# basins = flw.basins()
# target_basin = basins[row, col]
# catchment = basins == target_basin

# basin = flw.basins(xy=(snap_x, snap_y))
# results = [
#     shape(geom) for geom, val in shapes(catchment.astype("uint8"), transform=transform) if val == 1
# ]
# geom = flow_functions.remove_holes(unary_union(results))
# polygon = gpd.GeoDataFrame(geometry=[geom], crs=crs)

.\whitebox_tools.exe --run="FillDepressions" --dem='C:\Users\vanln\Downloads\Hydro-AI-Platform\inputs\dtm10.tif' --output='C:\Users\vanln\Downloads\Hydro-AI-Platform\filled.tif' --fix_flats -v --compress_rasters=False

******************************
* Welcome to FillDepressions *
* Powered by WhiteboxTools   *
* www.whiteboxgeo.com        *
******************************
Reading data...
Finding pit cells: 12%
Finding pit cells: 25%
Finding pit cells: 37%
Finding pit cells: 50%
Finding pit cells: 62%
Finding pit cells: 75%
Finding pit cells: 87%
Finding pit cells: 100%
Filling depressions: 0%
Filling depressions: 1%
Filling depressions: 2%
Filling depressions: 3%
Filling depressions: 4%
Filling depressions: 5%
Filling depressions: 6%
Filling depressions: 7%
Filling depressions: 8%
Filling depressions: 9%
Filling depressions: 10%
Filling depressions: 11%
Filling depressions: 12%
Filling depressions: 13%
Filling depressions: 14%
Filling depressions: 15%
Filling depressions: 16%
Filling de

0

In [58]:
with rasterio.open("filled.tif") as src:
    filled = src.read(1)

print(filled.shape)
print(filled.min(), filled.max())

RasterioIOError: filled.tif: No such file or directory

In [49]:
flow_acc > 50

array([[ True, False, False, ..., False, False, False],
       [False,  True,  True, ..., False, False, False],
       [False,  True,  True, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]], shape=(727, 1596))

In [40]:
polygon.to_file('test.geojson', driver='GeoJSON')

In [33]:
basin==1

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]], shape=(727, 1596))

## Process catchment

In [ ]:
catchment_UTM = catchment.to_crs(terrain.rio.crs)
# pour_point = gpd.GeoDataFrame({'geometry': [Point(6.355332891871, 62.458711715169)]}, crs=catchment.crs)
# pour_point = pour_point.to_crs(catchment_UTM.crs)
# pour_point
# catchment_UTM.total_bounds
# catchment_dir = os.path.normpath(os.path.join(test_folder, 'data/catchment'))
# if not os.path.exists(catchment_dir): os.makedirs(catchment_dir)
# catchment_UTM.to_file(os.path.join(catchment_dir, 'catchment.gpkg'), driver='GPKG')

array([  54235.02366958, 6953814.96627996,   67365.00444297,
       6957865.0110611 ])

## Clip dtm to catchment

In [ ]:
terrain_clipped = flow_functions.clip_catchment(catchment_UTM, terrain)
terrain_out_path = os.path.normpath(os.path.join(f'{test_folder}/data/dtm', "dtm_clipped.tif"))
terrain_clipped.rio.to_raster(terrain_out_path)

## Create merit hydro data

In [ ]:
with rasterio.open(terrain_path) as src:
    dem_array = src.read(1).astype(np.float32)
    profile = src.profile
    transform = src.transform
    crs = src.crs
NODATA_DEM, NODATA_INT = -9999.0, 0
profile.update(dtype=np.float32, nodata=NODATA_DEM)
# Fill depressions
elevtn_array, flwdir_array = dem.fill_depressions(
    elevtn=dem_array, nodata=NODATA_DEM, max_depth=-1
)
elevtn_array = np.where(np.isfinite(elevtn_array), elevtn_array, NODATA_DEM)
# Create flow direction
flw = pyflwdir.from_array(
    data=flwdir_array, ftype='d8', transform=transform, 
    latlon=crs.is_geographic
)
# Create slope
dx, dy = transform.a, abs(transform.e)
# Gradient elevation
dzdx = sobel(elevtn_array, axis=1, mode='nearest') / (8 * dx)
dzdy = sobel(elevtn_array, axis=0, mode='nearest') / (8 * dy)
slope_array = np.sqrt(dzdx**2 + dzdy**2)
slope_array = np.where(elevtn_array == NODATA_DEM, NODATA_DEM, slope_array).astype(np.float32)
# Create basins
basins_array = flw.basins()
# Create stream order
uparea_array = flw.upstream_area(unit='km2')
# Create stream mask and stream order
stream_mask = uparea_array > 30
strord_array = flw.stream_order(type='strahler', mask=stream_mask)
merit_dir = os.path.normpath(f'{test_folder}/data/merit_hydro')
if not os.path.exists(merit_dir): os.makedirs(merit_dir)
flow_functions.write_geotiff(elevtn_array, profile, os.path.join(merit_dir, 'elevtn.tif'))
profile_flwdir = {**profile, 'dtype': np.uint8, 'nodata': NODATA_INT}
flow_functions.write_geotiff(flwdir_array, profile_flwdir, os.path.join(merit_dir, 'flwdir.tif'))
profile_slope = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
flow_functions.write_geotiff(slope_array, profile_slope, os.path.join(merit_dir, 'lndslp.tif'))
profile_basins = {**profile, 'dtype': np.int32, 'nodata': NODATA_INT}
flow_functions.write_geotiff(basins_array, profile_basins, os.path.join(merit_dir, 'basins.tif'))
profile_uparea = {**profile, 'dtype': np.float32, 'nodata': NODATA_DEM}
flow_functions.write_geotiff(uparea_array, profile_uparea, os.path.join(merit_dir, 'uparea.tif'))
profile_strord = {**profile, 'dtype': np.int16, 'nodata': NODATA_INT}
flow_functions.write_geotiff(strord_array, profile_strord, os.path.join(merit_dir, 'strord.tif'))

## Prepare forcing data from the customized area

In [ ]:
# Read weather data
weather_path = os.path.join(sample_folder, 'alesund_weather.csv')
weather = pd.read_csv(weather_path, parse_dates=['datetime'], index_col='datetime')
weather_new = weather.loc['2025-01-01 00:00:00':'2025-01-10 00:00:00']

In [ ]:
# Create forcing nc file
time, crs = weather_new.index.to_numpy(), terrain.rio.crs
if crs is None: raise ValueError("Terrain has no crs")
ny, nx = terrain.rio.height, terrain.rio.width
forcing = {
    'precip': ['precip_mm', '(mm/h)'], 'temp': ['temp_C', '(degC)'],
    'kin': ['shortwave_Wm2', '(W/m^2)'], 'kout': ['longwave_Wm2', '(W/m^2)'],
    'wind': ['wind_mps', '(m/s)'], 'press_msl': ['pressure', '(Pa)']
}
forcing_dir = os.path.join(test_folder, 'data/forcing')
if not os.path.exists(forcing_dir): os.makedirs(forcing_dir)
out_path, datasets = os.path.join(forcing_dir, "my_forcing.nc"), {}
for item, values in forcing.items():
    data = weather_new[values[0]].values
    data_3d = create_forcing(time, ny, nx, data)
    datasets[item] = (('time', 'y', 'x'), data_3d, {'units': values[1]})
ds_final = xr.Dataset(
    data_vars=datasets, coords={"time": time, "y": terrain.y, "x": terrain.x}
)
ds_final.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=True)
ds_final.rio.write_crs(crs, inplace=True)
encoding = {
    var: {"zlib": True, "complevel": 4, "shuffle": True, "chunksizes": (1, 256, 256)}
    for var in ds_final.data_vars
}
ds_final.to_netcdf(out_path, engine='netcdf4', encoding=encoding)

## Process river data

In [30]:
# Create a random value for each river
river_UTM = river.to_crs(terrain.rio.crs)
cols = {'width': (0.05, 2), 'depth': (1, 5), 'manning_n': (0.03, 0.06)}
river_cols = river_UTM.columns.drop('geometry', errors='ignore')
for col in river_cols:
    river_UTM[col] = pd.to_numeric(river_UTM[col], errors='coerce')
for col, (low, high) in cols.items():
    river_mask = river_UTM[col].isna() | (river_UTM[col] == 'None')
    river_UTM.loc[river_mask, col] = np.round(np.random.uniform(low, high, river_mask.sum()), 3)
river_UTM = river_UTM[river_UTM.is_valid].reset_index(drop=True)

In [ ]:
print(np.isclose(river_UTM.total_bounds, terrain.rio.bounds()))

[  54815.00148636 6954445.00850605   67014.98469974 6957674.9541311 ]


(None, (53665.0, 6952895.0, 69625.0, 6960165.0))

In [54]:
# Process river
river_UTM["geometry"] = river_UTM.geometry.apply(lambda g: force_2d(g))
river_UTM = river_UTM.rename(columns={'width': 'rivwth', 'depth': 'rivdph'})
river_UTM = river_UTM[['rivwth', 'rivdph', 'manning_n', 'geometry']]
river_UTM.to_file(os.path.normpath(os.path.join(f'{test_folder}/data/river', 'river.gpkg')), driver='GPKG')

## Process soil data

In [ ]:
# Initialize variables
soil_types = {
    'clay': 'clyppt', 'sand': 'sndppt', 'silt': 'sltppt', 'bdod': 'bd', 'soc': 'oc', 'phh2o': 'ph'
}
depths = {
    '0-5cm_mean': 'sl1', '5-15cm_mean': 'sl2', '15-30cm_mean': 'sl3',
    '30-60cm_mean': 'sl4', '60-100cm_mean': 'sl5', '100-200cm_mean': 'sl6'
}
soil_dir = os.path.join(f'{test_folder}/data/soil')
if not os.path.exists(soil_dir): os.makedirs(soil_dir)
min_lon, min_lat, max_lon, max_lat = catchment.total_bounds
bbox = (float(min_lon), float(min_lat), float(max_lon), float(max_lat))

In [12]:
# Create soil thickness
with rasterio.open(terrain_path) as src:
    meta = src.meta.copy()
meta.update({"dtype": "float32", "nodata": -9999.0})
data = np.ones((meta["height"], meta["width"]), dtype="float32") * 100
data[data == meta["nodata"]] = 100
with rasterio.open(os.path.join(soil_dir, 'soilthickness.tif'), "w", **meta) as dst:
    dst.write(data, 1)

In [9]:
# Download soil data from ISRIC: https://files.isric.org/soilgrids/latest/data/
for item, name in tqdm(soil_types.items(), total=len(soil_types), desc='Downloading soil data'):
    wcs = WebCoverageService(f'https://maps.isric.org/mapserv?map=/map/{item}.map', version='1.0.0')
    for type, value in depths.items():
        idx = f'{item}_{type}'
        response = wcs.getCoverage(
            identifier=idx, crs='EPSG:4326', bbox=bbox,
            format='image/tiff', resx=0.0025, resy=0.0025
        )
        with MemoryFile(response.read()) as memfile:
            with memfile.open() as src:
                data = rioxarray.open_rasterio(src, masked=True)
                data_reprojected = data.rio.reproject_match(terrain)
            data_reprojected.rio.to_raster(os.path.join(soil_dir, f'{name}_{value}.tif'))

## Process land cover

In [15]:
dt = xr.open_zarr(r"C:\Users\vanln\.hydromt\artifact_data\latest\era5_hourly_zarr.zarr")
dt.close()

In [16]:
dt

<xarray.Dataset> Size: 567kB
Dimensions:      (time: 336, latitude: 7, longitude: 6)
Coordinates:
  * time         (time) datetime64[ns] 3kB 2010-02-01 ... 2010-02-14T23:00:00
  * latitude     (latitude) float32 28B 46.75 46.5 46.25 46.0 45.75 45.5 45.25
  * longitude    (longitude) float32 24B 11.75 12.0 12.25 12.5 12.75 13.0
Data variables:
    cape         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    d2m          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    kin          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    kout         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    precip       (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    press_msl    (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    spatial_ref  int32 4B ...
    tcwv         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    temp         (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    u10          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
    v10          (time, latitude, longitude) float32 56kB dask.array<chunksize=(336, 7, 6), meta=np.ndarray>
Attributes:
    category:        meteo
    history:         Extracted from Copernicus Climate Data Store
    paper_doi:       10.1002/qj.3803
    paper_ref:       Hersbach et al. (2019)
    source_license:  https://cds.climate.copernicus.eu/cdsapp/#!/terms/licenc...
    source_url:      https://doi.org/10.24381/cds.bd0915c6
    source_version:  ERA5 hourly data on pressure levels

In [ ]:
# Fix invalid land polygon
land_UTM = land.to_crs(terrain.rio.crs)
land_cols = ['land', 'LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
land_UTM = flow_functions.fix_invalid_polygon(land_UTM, land_cols)
land_layers = ['LAI', 'root_depth', 'interception', 'manning_n', 'albedo', 'kc']
for value in land_layers:
    land_path = os.path.normpath(os.path.join('test/data/landuse', f'{value}.tif'))
    write_tif(land_path, terrain, land_UTM, value)

In [30]:
# Create raster and lookup table (used for calibration)
land_class = land.to_crs(terrain.rio.crs).copy()
land_class['class'] = None
columns, table = np.unique(land_class['land'].values), {}
for id, item in enumerate(columns):
    temp = land_class[land_class['land'] == item]
    table[item] = np.float32(temp.iloc[0][land_layers].values)
    land_class.loc[temp.index, 'class'] = id
land_class = land_class[['class', 'geometry']]
land_class_path = os.path.normpath(os.path.join('test/data/lookup', 'land_classes.tif'))
write_tif(land_class_path, terrain, land_class, 'class')
# Create lookup table
lookup = pd.DataFrame.from_dict(table, orient='index', columns=land_layers)
lookup.index.name = 'landcover'
lookup.reset_index(inplace=True)
lookup.insert(0, 'class_id', lookup.index)
# Save lookup table to csv
lookup_csv_path = os.path.normpath(os.path.join('test/data/lookup', 'lookup_land.csv'))
lookup.to_csv(lookup_csv_path, index=False)

In [23]:
# Run HydroMT
model_path = os.path.normpath(f'{test_folder}/model')
if os.path.exists(model_path): shutil.rmtree(model_path)
!hydromt build wflow_sbm "./test/model" -i "./test/build.yml" -d "./test/config.yml" -v

2026-05-19 12:47:47,314 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-19 12:47:47,391 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from ./test/config.yml
2026-05-19 12:47:47,420 - hydromt.model.model - model - INFO - Initializing wflow_sbm model from hydromt_wflow (v1.0.1).
2026-05-19 12:47:47,420 - hydromt.data_catalog.data_catalog - data_catalog - INFO - Parsing data catalog from C:\Envs\hyd_ai\Lib\site-packages\hydromt_wflow\data\parameters_data.yml
2026-05-19 12:47:47,438 - hydromt.hydromt_wflow.wflow_base - wflow_base - INFO - Supported Wflow.jl version v1+
2026-05-19 12:47:47,439 - hydromt.hydromt_wflow.components.config - config - INFO - Reading default config file from C:/Envs/hyd_ai/Lib/site-packages/hydromt_wflow/data/wflow_sbm/wflow_sbm.toml.
2026-05-19 12:47:47,443 - hydromt - log - INFO - HydroMT version: 1.3.1
2026-05-19 12:47:47,444 - hydromt.model.model - model - INFO - build: setup_config
2026-05-19 12:47:47,444 - hydromt.m